In [1]:
from os.path import commonprefix
import attr
import json
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from argparse import ArgumentParser
from src.utils import load_df
import pandas as pd
from src.structs import LemmatizeRule
import attr
import pandas as pd
from src.categories import (
    name2mapping_to_id,
    name2mapping_from_id,
    names_order,
    UNDEFINED
)
from collections import Counter
import numpy as np

from src.structs import LemmatizeRule
from src.utils import load_df
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.tree import export_text

In [2]:
data_path = Path("/mnt/data_storage/datasets/generative_lemmatization_datasets/csvs")

In [3]:
# в test берем dev сплит чтобы в замерах финального пайплайна на настоящем test сплите в других работах не было утечки
df_train = load_df(data_path / "train.csv", dedup=True, drop_hapax=True)
df_test = load_df(data_path / "dev.csv", dedup=True, drop_hapax=True)

df_train = df_train.drop(["subset", "filename"], axis=1)
df_test = df_test.drop(["subset", "filename"], axis=1)

In [4]:
df_train["feats"] = df_train["feats"].fillna("No Feats")
df_test["feats"] = df_test["feats"].fillna("No Feats")

In [5]:
# у теста и трейна есть пересечения (высокочастотные слова и пунктуация). 
# поэтому из test дополнитлеьно уберем то что есть в train

cols = ['form', 'pos', 'feats', 'lemma']

key_train = df_train[cols].astype(str).agg("|".join, axis=1)
key_test = df_test[cols].astype(str).agg("|".join, axis=1)
mask = key_test.isin(key_train)
df_test = df_test[~mask]

In [6]:
print(f"Test size: {df_test.shape[0]}")

Test size: 4827


In [7]:
df_test.head()

,form,pos,feats,lemma
3,Ставь,VERB,Aspect:Imp Mood:Imp Number:Sing Person:2 Trans...,ставить
4,❤️,SYM,No Feats,❤️
35,собираем,VERB,Aspect:Imp Mood:Ind Number:Plur Person:1 Tense...,собирать
51,подкинуть,VERB,Aspect:Perf Transit:Tran VerbForm:Inf Voice:Act,подкинуть
112,веселые,ADJ,Animacy:Inan Case:Acc Degree:Pos Number:Plur,веселый


In [8]:
df_train.info()

<class 'pandas.DataFrame'>
Index: 106775 entries, 0 to 2148717
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   form    106775 non-null  str  
 1   pos     106775 non-null  str  
 2   feats   106775 non-null  str  
 3   lemma   106775 non-null  str  
dtypes: str(4)
memory usage: 4.1 MB


In [9]:
df_test.info()

<class 'pandas.DataFrame'>
Index: 4827 entries, 3 to 254911
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   form    4827 non-null   str  
 1   pos     4827 non-null   str  
 2   feats   4827 non-null   str  
 3   lemma   4827 non-null   str  
dtypes: str(4)
memory usage: 188.6 KB


In [10]:
## таким образом в known test split попала частотная лексика, которая естественным образом появляется и в трейне (из-за св-в естественного языка и речи)
## а в unknown test split попадают менее частотные случаи

In [11]:
## TODO: предобработка архаики? замена ятей, удаление терминалных твердых знаков и тд
## see https://github.com/ingoboerner/russpelling
## see https://sites.google.com/site/seesscm/pre-reform-russian-orthography-cheat-sheet?pli=1&authuser=0

In [ ]:
# from https://github.com/DanAnastasyev/GramEval2020/blob/master/solution/train/lemmatize_helper.py
def predict_lemmatize_rule(word: str, lemma: str):
    def _predict_lemmatize_rule(word: str, lemma: str, **kwargs):
        if len(word) == 0:
            return LemmatizeRule(append_suffix=lemma, **kwargs)

        common_prefix = commonprefix([word, lemma])
        if len(common_prefix) == 0:
            rule = _predict_lemmatize_rule(word[1:], lemma, **kwargs)
            return attr.evolve(rule, cut_prefix=rule.cut_prefix + 1)

        return LemmatizeRule(cut_suffix=len(word) - len(common_prefix),
                                append_suffix=lemma[len(common_prefix):], **kwargs)

    word, lemma = word.replace('ё', 'е'), lemma.replace('ё', 'е')
    return min([
        _predict_lemmatize_rule(word, lemma),
        _predict_lemmatize_rule(word.lower(), lemma, lower=True),
        _predict_lemmatize_rule(word.capitalize(), lemma, capitalize=True),
        _predict_lemmatize_rule(word.upper(), lemma, upper=True)
    ], key=lambda rule: rule.cut_prefix + rule.cut_suffix)


def lemmatize(word: str, rule: LemmatizeRule):

    assert isinstance(rule, LemmatizeRule)

    if rule.lower:
        word = word.lower()
    if rule.capitalize:
        word = word.capitalize()
    if rule.upper:
        word = word.upper()

    if rule.cut_suffix != 0:
        lemma = word[rule.cut_prefix: -rule.cut_suffix]
    else:
        lemma = word[rule.cut_prefix:]
    lemma += rule.append_suffix

    return lemma


In [13]:
# предскажем правила лемматизации для всех слов, но работать будем только с правилами из трейна

tqdm.pandas(desc="Подготовка правил лемматизации")

for df in (df_train, df_test):
    def _foo(row):
        return predict_lemmatize_rule(row["form"], row["lemma"])
    df["lemmatization_rule"] = df.progress_apply(
        _foo,
        axis=1
    )

Подготовка правил лемматизации: 100%|██████████| 4827/4827 [00:00<00:00, 34178.26it/s]


In [14]:
unique_rules = set(df_train["lemmatization_rule"].unique())
len(unique_rules)

1816

In [15]:
def _sort_rules(rules):
    return sorted(
        rules, key=lambda r: (
            r.cut_prefix, r.cut_suffix, r.append_suffix, r.lower, r.capitalize, r.upper
        )
    )

In [16]:
id2rule: dict[int, LemmatizeRule]= {
        i: rule for i, rule
        in enumerate(_sort_rules(unique_rules))
    }
rule2id = {rule: i for i, rule in id2rule.items()}

In [17]:
for df in (df_train, df_test):
    df["rule_id"] = df["lemmatization_rule"].map(rule2id)

In [18]:
# почти для всех случаев правила определяются в трейне!
# неопределенные случаи надо заполнить и заценить, что там
df_test.info()

<class 'pandas.DataFrame'>
Index: 4827 entries, 3 to 254911
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   form                4827 non-null   str    
 1   pos                 4827 non-null   str    
 2   feats               4827 non-null   str    
 3   lemma               4827 non-null   str    
 4   lemmatization_rule  4827 non-null   object 
 5   rule_id             4746 non-null   float64
dtypes: float64(1), object(1), str(4)
memory usage: 264.0+ KB


In [19]:
UNDEFINED_RULE_ID = -100

In [20]:
df_test["rule_id"] = df_test["rule_id"].fillna(UNDEFINED_RULE_ID)

In [21]:
unkn_test_undefined_rule = df_test[df_test["rule_id"] == UNDEFINED_RULE_ID]

In [22]:
unkn_test_undefined_rule.shape[0]

81

In [23]:
unkn_test_undefined_rule.head(15)

,form,pos,feats,lemma,lemmatization_rule,rule_id
3552,подольше,ADV,Degree:Cmp2,долго,"LemmatizeRule(cut_prefix=2, cut_suffix=3, appe...",-100.0
6465,гложет,VERB,Aspect:Imp Mood:Ind Number:Sing Person:3 Tense...,глодать,"LemmatizeRule(cut_prefix=0, cut_suffix=3, appe...",-100.0
10097,Апрѣля,NOUN,Animacy:Inan Case:Gen Gender:Masc Number:Sing,апрель,"LemmatizeRule(cut_prefix=0, cut_suffix=3, appe...",-100.0
10394,вѣрностію,NOUN,Animacy:Inan Case:Ins Gender:Fem Number:Sing,верность,"LemmatizeRule(cut_prefix=0, cut_suffix=8, appe...",-100.0
10635,изстари,ADV,Degree:Pos,исстари,"LemmatizeRule(cut_prefix=0, cut_suffix=6, appe...",-100.0
10702,помѣстья,NOUN,Animacy:Inan Case:Acc Gender:Neut Number:Plur,поместье,"LemmatizeRule(cut_prefix=0, cut_suffix=5, appe...",-100.0
11451,вѣрноподданнымъ,ADJ,Case:Dat Degree:Pos Number:Plur,верноподданный,"LemmatizeRule(cut_prefix=0, cut_suffix=14, app...",-100.0
12189,наслѣдственныхъ,ADJ,Case:Gen Degree:Pos Number:Plur,наследственный,"LemmatizeRule(cut_prefix=0, cut_suffix=11, app...",-100.0
12367,Намѣстничествѣ,NOUN,Animacy:Inan Case:Loc Gender:Neut Number:Sing,наместничество,"LemmatizeRule(cut_prefix=0, cut_suffix=11, app...",-100.0
12802,Генералъ-Губернатора,NOUN,Animacy:Anim Case:Gen Gender:Masc Number:Sing,генерал-губернатор,"LemmatizeRule(cut_prefix=0, cut_suffix=13, app...",-100.0


In [24]:
# доля случаев с неопределенным парвилом (правило не встретилось в тренировочных данных)
# часто это связвно с дореформенной орфографией. возможно есть смысл добавить правило на ее нормализацию, см выше.
unkn_test_undefined_rule.shape[0] / df_test.shape[0]

0.01678060907395898

In [25]:
## top k most frequent rules in train

k = 10
topk_rule_ids = df_train['rule_id'].value_counts().head(k).rename_axis('rule_id').reset_index(name='count')

for row in topk_rule_ids.itertuples(index=False):
    print(f"\n{'='*80}")
    print(f"rule_id: {row.rule_id}  |  count: {row.count}")
    print(f"rule: {id2rule[row.rule_id]}")
    subset = df_train[df_train['rule_id'] == row.rule_id]
    if set(subset["pos"]) == set(df_train["pos"]):
        pos_set = "ALL TAGS!!!"
    else:
        pos_set = sorted(subset['pos'].dropna().unique())
    print(f"pos tags: {pos_set}")
    print('-'*80)
    ex = subset.sample(
        n=min(5, row.count), random_state=42
    )
    for _, r in ex.iterrows():
        print(f"  form={r['form']!r:15}  lemma={r['lemma']!r:15}  pos={r['pos']!r:10}  feats={r['feats']}")


rule_id: 0  |  count: 23207
rule: LemmatizeRule(cut_prefix=0, cut_suffix=0, append_suffix='', lower=False, capitalize=False, upper=False)
pos tags: ALL TAGS!!!
--------------------------------------------------------------------------------
  form='запоминание'    lemma='запоминание'    pos='NOUN'      feats=Animacy:Inan Case:Acc Gender:Neut Number:Sing
  form='пристрастие'    lemma='пристрастие'    pos='NOUN'      feats=Animacy:Inan Case:Acc Gender:Neut Number:Sing
  form='Франсуа'        lemma='Франсуа'        pos='PROPN'     feats=Animacy:Anim Case:Nom Gender:Masc InflClass:Ind NameType:Giv Number:Sing
  form='равенство'      lemma='равенство'      pos='NOUN'      feats=Animacy:Inan Case:Acc Gender:Neut Number:Sing
  form='макрух'         lemma='макрух'         pos='X'         feats=Foreign:Yes

rule_id: 61  |  count: 9229
rule: LemmatizeRule(cut_prefix=0, cut_suffix=1, append_suffix='', lower=False, capitalize=False, upper=False)
pos tags: ['ADJ', 'ADP', 'ADV', 'ADVPRO', 'ANUM', '

In [26]:
def get_pref(row):
    return row["form"][:row["lemmatization_rule"].cut_prefix]

In [27]:
all_prefs = []
all_pref_rules = []

for _, row in df_train.iterrows():
    pref = get_pref(row)
    if pref:
        all_prefs.append(pref)
        all_pref_rules.append(row["lemmatization_rule"])
Counter(all_prefs).most_common(20)

[('по', 43),
 ('ща', 14),
 ('есть', 7),
 ('им', 7),
 ('ег', 6),
 ('нег', 4),
 ('ней', 4),
 ('Есть', 4),
 ('хуже', 4),
 ('чем', 4),
 ('нихъ', 4),
 ('на', 3),
 ('их', 3),
 ('нас', 3),
 ('лучше', 3),
 ('Лучше', 3),
 ('них', 3),
 ('больше', 3),
 ('ним', 3),
 ('На', 3)]

In [28]:
len(all_prefs), len(Counter(all_prefs))

(413, 238)

In [29]:
all_prefs_lens = [len(p) for p in all_prefs]

In [30]:
np.quantile(all_prefs_lens, (0.5, 0.75, 0.95, 0.99, 1))

array([ 3.,  5.,  7.,  8., 11.])

In [31]:
## случаи с prefix выглядят скорее как шум. можно их убрать из трейна в принципе. в тесте оставим и потом проверим, влияет ли на что-то.
Counter(all_pref_rules).most_common(10)

[(LemmatizeRule(cut_prefix=2, cut_suffix=0, append_suffix='она', lower=False, capitalize=False, upper=False),
  18),
 (LemmatizeRule(cut_prefix=4, cut_suffix=0, append_suffix='быть', lower=False, capitalize=False, upper=False),
  15),
 (LemmatizeRule(cut_prefix=3, cut_suffix=0, append_suffix='они', lower=False, capitalize=False, upper=False),
  13),
 (LemmatizeRule(cut_prefix=5, cut_suffix=0, append_suffix='ребенок', lower=False, capitalize=False, upper=False),
  10),
 (LemmatizeRule(cut_prefix=3, cut_suffix=0, append_suffix='она', lower=False, capitalize=False, upper=False),
  10),
 (LemmatizeRule(cut_prefix=5, cut_suffix=0, append_suffix='никакой', lower=False, capitalize=False, upper=False),
  10),
 (LemmatizeRule(cut_prefix=3, cut_suffix=0, append_suffix='он', lower=False, capitalize=False, upper=False),
  8),
 (LemmatizeRule(cut_prefix=2, cut_suffix=0, append_suffix='н', lower=False, capitalize=False, upper=False),
  8),
 (LemmatizeRule(cut_prefix=2, cut_suffix=0, append_suffix='_

In [32]:
prefix_rules_ids = [i for i, rule in id2rule.items() if rule.cut_prefix]

In [33]:
df_train = df_train[~df_train["rule_id"].isin(prefix_rules_ids)]

In [34]:
## мы избавились в трейне от правил с префиксами, поэтому можем позволить себе кодировать только суффиксы
## здесь же сделаем небольшую предобработку: переведем е в ё
## TODO: проверить есть ли смысл делать form.lower()
for n in range(1, 4):
    colname = f"suf{n}ch"
    df_train[colname] = (
        df_train["form"]
        .str.replace("ё", "е")
        # .str.lower()
        .apply(lambda s: "<pad>"*max(n - len(s), 0) + s[-n:])
    )

# df_train["istitle"] = df["form"].str.istitle()
# df_train["isupper"] = df["form"].str.isupper()



In [35]:
## гапаксы вроде убраны, а все равно матрицы будут огромные
for col in ("suf1ch", "suf2ch", "suf3ch",):
    print(col, df_train[col].nunique())


# без предобработки (е/ё):
# suf1ch 192
# suf2ch 1412
# suf3ch 6608

suf1ch 191
suf2ch 1388
suf3ch 6501


In [36]:
def parse_feats(feats: str, names_order=names_order, name2mapping_to_id=name2mapping_to_id) -> dict:

    result = {name: UNDEFINED for name in names_order}

    if not isinstance(feats, str) or feats == "No Feats":
        return result

    for part in filter(lambda elem: ":" in elem, feats.split()):
        k, v = part.split(":", 1)

        if k in result and v in name2mapping_to_id[k]:
            result[k] = v

    return result

feats_dicts = [parse_feats(feats) for feats in df_train["feats"].tolist()]
feats_df = pd.DataFrame(feats_dicts, index=df_train.index)
feats_df["upos"] = df_train["pos"]

df_train = pd.concat([df_train, feats_df], axis=1)

In [37]:
df_train.head()

,form,pos,feats,lemma,lemmatization_rule,rule_id,suf1ch,suf2ch,suf3ch,upos,...,NumType,Variant,PronType,Abbr,Poss,InflClass,Reflex,Polarity,Foreign,Hyph
0,Снова,ADV,Degree:Pos,снова,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",3,а,ва,ова,ADV,...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]
1,приобрел,VERB,Aspect:Perf Gender:Masc Mood:Ind Number:Sing T...,приобрести,"LemmatizeRule(cut_prefix=0, cut_suffix=1, appe...",109,л,ел,рел,VERB,...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]
2,дозу,NOUN,Animacy:Inan Case:Acc Gender:Fem Number:Sing,доза,"LemmatizeRule(cut_prefix=0, cut_suffix=1, appe...",64,у,зу,озу,NOUN,...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]
3,",",PUNCT,No Feats,",","LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",0,",","<pad>,","<pad><pad>,",PUNCT,...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]
4,В,ADP,No Feats,в,"LemmatizeRule(cut_prefix=0, cut_suffix=0, appe...",3,В,<pad>В,<pad><pad>В,ADP,...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]


In [38]:
df_train.columns

Index(['form', 'pos', 'feats', 'lemma', 'lemmatization_rule', 'rule_id',
       'suf1ch', 'suf2ch', 'suf3ch', 'upos', 'Animacy', 'Case', 'Gender',
       'Number', 'NameType', 'Aspect', 'Mood', 'Tense', 'Transit', 'VerbForm',
       'Voice', 'Degree', 'Person', 'NumForm', 'NumType', 'Variant',
       'PronType', 'Abbr', 'Poss', 'InflClass', 'Reflex', 'Polarity',
       'Foreign', 'Hyph'],
      dtype='str')

In [39]:
G_cols = ["suf1ch", "suf2ch", "suf3ch"]
M_cols = ['upos', 'Animacy', 'Case', 'Gender', 'Number', 'NameType',
       'Aspect', 'Mood', 'Tense', 'Transit', 'VerbForm', 'Voice', 'Degree',
       'Person', 'NumForm', 'NumType', 'Variant', 'PronType', 'Abbr', 'Poss',
       'InflClass', 'Reflex', 'Polarity', 'Foreign', 'Hyph']
tgt_col = ["rule_id"]

In [48]:
def get_rules(df: pd.DataFrame, cols:list[str], tgt_col: list[str]):

    # ACHTUNG! tgt_col should have 1 element only
    assert len(tgt_col) == 1

    enc = OneHotEncoder(handle_unknown='error', sparse_output=False)

    # будем дропать дубликаты, важно дропнуть и по таргетному правилу тоже
    # теперь работаем с уникальными строками, которые мапятся на разные правила.
    # таким образом можно попробовать найти подмножества фич из них, которые детерминированно (или почти) мапятся на одно и то же правило
    df = df[cols + tgt_col].drop_duplicates()
    print(f"Final train size: {df.shape[0] - 1}")

    X = enc.fit_transform(df[cols])
    y = df[tgt_col]

    tree = DecisionTreeClassifier(
        splitter="best",
        max_depth=10,     
        min_samples_leaf=20,
        min_samples_split=40,
        max_features=None,
        class_weight=None,
        random_state=42,
    )

    # TODO: подбор параметров
    tree.fit(X, y)

    rules = export_text(tree, feature_names=list(enc.get_feature_names_out(cols)))
    return rules

In [49]:
M = get_rules(df_train, M_cols, tgt_col)

Final train size: 8222


In [50]:
print(M)

|--- InflClass_Ind <= 0.50
|   |--- VerbForm_Part <= 0.50
|   |   |--- VerbForm_[UND] <= 0.50
|   |   |   |--- Voice_Mid <= 0.50
|   |   |   |   |--- Tense_Past <= 0.50
|   |   |   |   |   |--- Aspect_[UND] <= 0.50
|   |   |   |   |   |   |--- Person_3 <= 0.50
|   |   |   |   |   |   |   |--- Number_Plur <= 0.50
|   |   |   |   |   |   |   |   |--- VerbForm_Inf <= 0.50
|   |   |   |   |   |   |   |   |   |--- Mood_Ind <= 0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 117
|   |   |   |   |   |   |   |   |   |--- Mood_Ind >  0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 351
|   |   |   |   |   |   |   |   |--- VerbForm_Inf >  0.50
|   |   |   |   |   |   |   |   |   |--- Transit_Tran <= 0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |   |   |--- Transit_Tran >  0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |--- Number_Plur >  0.50
|   |   |   |   |   |   |   |   |--- Person_2 <= 0.50
|  

In [ ]:
# VerbForm_Inf --> 0
# InflClass_Ind --> 0
# VerbForm_Part --> 895

In [55]:
id2rule[895]

LemmatizeRule(cut_prefix=0, cut_suffix=4, append_suffix='ть', lower=False, capitalize=False, upper=False)

In [58]:
lemmatize("живешь", id2rule[895])

'жить'

In [ ]:
# плохое правило
df_train[df_train["feats"].str.contains("VerbForm:Part")]["rule_id"].value_counts()

rule_id
895     1036
351      280
526      273
1086     234
1370     190
        ... 
1502       1
1613       1
1647       1
1302       1
1117       1
Name: count, Length: 170, dtype: int64

In [51]:
G = get_rules(df_train, G_cols, tgt_col)

Final train size: 15159


In [52]:
print(G)

|--- suf1ch_у <= 0.50
|   |--- suf1ch_ы <= 0.50
|   |   |--- suf1ch_ъ <= 0.50
|   |   |   |--- suf1ch_е <= 0.50
|   |   |   |   |--- suf1ch_и <= 0.50
|   |   |   |   |   |--- suf1ch_ю <= 0.50
|   |   |   |   |   |   |--- suf1ch_я <= 0.50
|   |   |   |   |   |   |   |--- suf1ch_а <= 0.50
|   |   |   |   |   |   |   |   |--- suf1ch_м <= 0.50
|   |   |   |   |   |   |   |   |   |--- suf1ch_ѣ <= 0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |   |   |--- suf1ch_ѣ >  0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 64
|   |   |   |   |   |   |   |   |--- suf1ch_м >  0.50
|   |   |   |   |   |   |   |   |   |--- suf2ch_ум <= 0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |   |   |--- suf2ch_ум >  0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |--- suf1ch_а >  0.50
|   |   |   |   |   |   |   |   |--- suf2ch_ла <= 0.50
|   |   |   |   |   |   |   |   |   |--- suf2ch_

In [53]:
GM = get_rules(df_train, G_cols + M_cols, tgt_col)

Final train size: 27687


In [54]:
print(GM)

|--- Transit_[UND] <= 0.50
|   |--- suf1ch_л <= 0.50
|   |   |--- suf1ch_в <= 0.50
|   |   |   |--- Case_[UND] <= 0.50
|   |   |   |   |--- Reflex_[UND] <= 0.50
|   |   |   |   |   |--- Tense_Past <= 0.50
|   |   |   |   |   |   |--- Gender_[UND] <= 0.50
|   |   |   |   |   |   |   |--- class: 1370
|   |   |   |   |   |   |--- Gender_[UND] >  0.50
|   |   |   |   |   |   |   |--- class: 1316
|   |   |   |   |   |--- Tense_Past >  0.50
|   |   |   |   |   |   |--- class: 1370
|   |   |   |   |--- Reflex_[UND] >  0.50
|   |   |   |   |   |--- Aspect_Perf <= 0.50
|   |   |   |   |   |   |--- Tense_Past <= 0.50
|   |   |   |   |   |   |   |--- Transit_Intr <= 0.50
|   |   |   |   |   |   |   |   |--- Case_Gen <= 0.50
|   |   |   |   |   |   |   |   |   |--- Case_Nom <= 0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 895
|   |   |   |   |   |   |   |   |   |--- Case_Nom >  0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 895
|   |   |   |   |   |   |   |   |--- Case_Gen >  

In [43]:
def check_InflClass_Ind_rule(df):
    return df[(df["feats"].str.contains("InflClass:Ind")) & (df["rule_id"] != 0) & (df["form"].str.lower() != df["lemma"].str.lower())]

In [44]:
check_InflClass_Ind_rule(df_train)

,form,pos,feats,lemma,lemmatization_rule,rule_id,suf1ch,suf2ch,suf3ch,upos,...,NumType,Variant,PronType,Abbr,Poss,InflClass,Reflex,Polarity,Foreign,Hyph
178421,Джефри,PROPN,Animacy:Anim Case:Nom Gender:Masc InflClass:In...,Джеффри,"LemmatizeRule(cut_prefix=0, cut_suffix=2, appe...",357,и,ри,фри,PROPN,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]
668708,Мерфи,PROPN,Animacy:Anim Case:Gen Gender:Masc InflClass:In...,Мэрфи,"LemmatizeRule(cut_prefix=0, cut_suffix=4, appe...",920,и,фи,рфи,PROPN,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]
1537394,Пинокио,PROPN,Animacy:Anim Case:Gen Gender:Masc InflClass:In...,Пиноккио,"LemmatizeRule(cut_prefix=0, cut_suffix=2, appe...",253,о,ио,кио,PROPN,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]
1537550,Пинокио,PROPN,Animacy:Anim Case:Loc Gender:Masc InflClass:In...,Пиноккио,"LemmatizeRule(cut_prefix=0, cut_suffix=2, appe...",253,о,ио,кио,PROPN,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]
1578212,тысячу,NOUN,Animacy:Inan Case:Acc Gender:Fem InflClass:Ind...,тысяча,"LemmatizeRule(cut_prefix=0, cut_suffix=1, appe...",64,у,чу,ячу,NOUN,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]


In [45]:
check_InflClass_Ind_rule(df_test)

,form,pos,feats,lemma,lemmatization_rule,rule_id
